In [ ]:
# 1. Load Dataset 

# Import libraries 
import pandas as pd 
import numpy as np

# Load dataset (downloaded from Kaggle)
df = pd.read_csv("heart_disease_uci.csv")

# Print first 5 rows
print("First 5 rows:")
display(df.head()) 


In [ ]:
# 2. Data Cleaning

# Fix numeric conversions 
# Look for columns that should be numeric but are object 
for col in df.columns:
    if df[col].dtype == "object":
        try:
            df[col] = pd.to_numeric(df[col], errors="raise")
        except:
            # If conversion fails, leave it as object (it's categorical)
            pass

# Drop unnecessary/sparse columns
df.drop(columns=["fbs", "exang", "ca", "thal"], inplace=True, errors="ignore")

# Impute median for continuous variables 
for col in ["trestbps", "chol", "thalch"]:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val) 

# oldpeak: drop missing rows, fix negatives 
df = df.dropna(subset=["oldpeak"])
df["oldpeak"] = df["oldpeak"].abs()

# slope: drop rows where missing 
df = df.dropna(subset=["slope"])

# Reset index after cleaning
df.reset_index(drop=True, inplace=True) 


In [ ]:
# 3. Save Cleaned Dataset

df.to_csv("heart_disease_cleaned.csv", index=False)
print("Cleaned dataset saved as 'heart_disease_cleaned.csv' in your current folder.")


In [ ]:
# 4. Preprocessing 

from sklearn.preprocessing import MinMaxScaler 
from sklearn.model_selection import train_test_split 

# Select features 

numeric_features = ["age", "trestbps", "chol", "thalch", "oldpeak"]
categ_features = ["sex", "cp", "restecg", "slope"]

X = df[numeric_features + categ_features].copy()
y = df["num"].apply(lambda x: 1 if x > 0 else 0)

# Convert to float
X[numeric_features] = X[numeric_features].astype(float)

# Scale numeric features
scaler = MinMaxScaler()
X[numeric_features] = scaler.fit_transform(X[numeric_features])

# Encode categorical features 
X = pd.get_dummies(X, columns=categ_features, drop_first=True)

# Train-test split 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


In [ ]:
# 5. Model Comparison 

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:,1] if hasattr(model, "predict_proba") else None
    
    print("Classification Report:")
    print(classification_report(y_test, y_pred))
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    if y_prob is not None:
        print("ROC-AUC Score:", round(roc_auc_score(y_test, y_prob), 3))
    print("\n")


In [ ]:
# 6. EDA Section 

import matplotlib.pyplot as plt
import seaborn as sns

# Basic Statistical Analysis 

numeric_cols = df.select_dtypes(include=["int64","float64"]).columns 

for col in numeric_cols:
    print(f"\nColumn:{col}") # shows column name 
    print("Mean:", df[col].mean())
    print("Median:", df[col].median())
    print("Mode:", df[col].mode()[0])
    print("Variance:", df[col].var())
    print("Standard Deviation:", df[col].std()) 

# Pie Chart for Gender Distribution 
gender_counts = df['sex'].value_counts()
plt.figure(figsize=(5,5))
plt.pie(gender_counts, labels=['Male', 'Female'], autopct='%1.1f%%', colors=['skyblue', 'lightpink'])
plt.title("Gender Distribution")
plt.show()

# Pie Chart for Disease Distribution 
target_counts = df['num'].apply(lambda x: 1 if x>0 else 0).value_counts()
plt.figure(figsize=(5,5))
plt.pie(target_counts, labels=['No Disease', 'Disease'], autopct='%1.1f%%', colors=['lightgreen', 'salmon'])
plt.title("Heart Disease Distribution")
plt.show()

# Histograms of Numeric Features
numeric_features = ["age", "trestbps", "chol", "thalch", "oldpeak"]
df[numeric_features].hist(figsize=(12,6), bins=15, color='skyblue', edgecolor='black')
plt.suptitle("Histograms of Numeric Features")
plt.show()

# Bar Charts for Categorical Features
categorical_features = ["cp", "restecg", "slope"]
for col in categorical_features:
    plt.figure(figsize=(6,4))
    sns.countplot(data=df, x=col, palette='pastel')
    plt.title(f"Bar Chart of {col}")
    plt.show() 

# Scatter Plots for Relationships with Disease Status 
plt.figure(figsize=(6,4))
plt.scatter(df['age'], df['thalch'], c=df['num'].apply(lambda x: 1 if x>0 else 0), cmap='coolwarm', alpha=0.7)
plt.xlabel("Age")
plt.ylabel("Max Heart Rate (thalch)")
plt.title("Age vs Max Heart Rate by Disease Status")
plt.show()

plt.figure(figsize=(6,4))
plt.scatter(df['chol'], df['oldpeak'], c=df['num'].apply(lambda x: 1 if x>0 else 0), cmap='coolwarm', alpha=0.7)
plt.xlabel("Cholesterol (chol)")
plt.ylabel("ST Depression (oldpeak)")
plt.title("Cholesterol vs ST Depression by Disease Status")
plt.show() 
